# 2022_12_16



### weather data error

* 10년도 파일 

|  | day | hour | value | month |
|---:|---:|---:|---:|---:|
| 0 | Start : 20130101 | NaN | 0.0 | 1.0 |
| 1 | 1 | 0200 | -5.0 | 1.0 |
| 2 | 1 | 0200 | -6.0 | 1.0 |
| 3 | 1 | 0200 | -8.0 | 1.0 |
| 4 | 1 | 0200 | -8.9 | 1.0 |
| ... | ... | ... | ... | ... |
| 96379 | 31 | 2300 | -5.3 | 12.0 |
| 96380 | 31 | 2300 | -6.3 | 12.0 |
| 96381 | 31 | 2300 | -6.0 | 12.0 |
| 96382 | 31 | 2300 | -5.0 | 12.0 |
| 96383 | 31 | 2300 | 1.1 | 12.0 |


* 13년도 파일 

| day | day | hour | value | month |
|---:|---:|---:|---:|---:|
| 0 | Start : 20130101 | NaN | NaN | 1.0 |
| 1 | 1 | 0200 | -5.000000 | 1.0 |
| 2 | 1 | 0200 | -6.000000 | 1.0 |
| 3 | 1 | 0200 | -8.000000 | 1.0 |
| 4 | 1 | 0200 | -8.900000 | 1.0 |
| ... | ... | ... | ... | ... |
| 96379 | 31 | 2300 | -5.300000 | 12.0 |
| 96380 | 31 | 2300 | -6.300000 | 12.0 |
| 96381 | 31 | 2300 | -6.000000 | 12.0 |
| 96382 | 31 | 2300 | -5.000000 | 12.0 |
| 96383 | 31 | 2300 | 1.100000 | 12.0 |


value의 값이 다르게 나와서 파일을 건드리는데 코드에서 에러가 남  
확인해보니 13년도 파일에   
`  format: day,hour,forecast,value  location:73_134 Start : 20160101`  
같은 row가 추가 되어서 pandas가 숫자로 인식못하고 str로 인식해서 이런 문제가 생기는 듯 함   
-> str이 있느지 검사해서 str이 있다면 이 row을 없애주는 cleaning 작업 수행  

### prediction error

기존에는 dataloader를 사용하지 않고 그냥 raw data를 TFT 모델에 집어넣어서 에러가 남  
``` python
    predition= np.array([])
    for file in sorted(os.listdir('../../val'))[1:-1]:
        val_data = data_processing('../../val/' + file , factor , ewma_funs)
        pred , x, idx_df = tft.predict(val_dataloader, mode='raw', return_x = True , return_index = True)
        idx = idx_df[idx_df['h_dong'] == dong].index[0]
        predition = np.concatenate([predition,pred['prediction'][idx, : , n]])
```

dataloader가 scaling 및 normalization의 작업을 수행하는 데 이러한 작업이 없어서 에러가 나타난 건줄 알았음    
dataloader를 사용하지 않는다면 train data와 test data간의 scaling의 오류가 나타날 수 있음   

```python
    predition= np.array([])
    for file in sorted(os.listdir('../../val'))[1:-1]:
        val_data = data_processing('../../val/' + file , factor , ewma_funs)
        validation = TimeSeriesDataSet.from_dataset(training, val_data, predict=True, stop_randomization=True)
        val_dataloader = validation.to_dataloader(train=False, batch_size=16, num_workers=0)
        pred , x, idx_df = tft.predict(val_dataloader, mode='raw', return_x = True , return_index = True)
        idx = idx_df[idx_df['h_dong'] == dong].index[0]
        predition = np.concatenate([predition,pred['prediction'][idx, : , n]])
```

실제 원인은 다음과 같음  
* 일주일 단위로 기준을 했을때 사건이 발생하지 않는 경우
* 위 경우에는 ewma의 scale up 과정에서 ewma의 최대값으로 나누고 원래 데이터의 최대값을 곱해주는데 여기서 ewma가 최대값인 0으로 나눠주므로 Nan이 발생함 
* fillna(0)을 넣어줌으로써 해결함  
```python
    predition= np.array([])
    for file in sorted(os.listdir('../../val'))[1:-1]:
        val_data = data_processing('../../val/' + file , factor , ewma_funs)
        val_data = val_data.fillna(0)
        validation = TimeSeriesDataSet.from_dataset(training, val_data, predict=True, stop_randomization=True)
        val_dataloader = validation.to_dataloader(train=False, batch_size=16, num_workers=0)
        pred , x, idx_df = tft.predict(val_dataloader, mode='raw', return_x = True , return_index = True)
        idx = idx_df[idx_df['h_dong'] == dong].index[0]
        predition = np.concatenate([predition,pred['prediction'][idx, : , n]])
```

### time 수정 

```python
def prediction_plot_n(training, tft ,data, dong , title, ewma_funs, factor ,n):
    fig = plt.figure(figsize=(25, 7))
    df = data[data['h_dong']==dong]
    df_index = df[df['time_idx'] > 24* 7 -1]['REG_DTIME']
    org_count_ewma = df[df['time_idx'] > 24* 7 -1 ]['count']

    prediction_df = pd.DataFrame()
    for file in sorted(os.listdir('../../val'))[1:]:
        val_data = data_processing('../../val/' + file , factor , ewma_funs)
        val_data = val_data.fillna(0)
        validation = TimeSeriesDataSet.from_dataset(training, val_data, predict=True, stop_randomization=True)
        val_dataloader = validation.to_dataloader(train=False, batch_size=16, num_workers=0)
        pred , x, idx_df = tft.predict(val_dataloader, mode='raw', return_x = True , return_index = True)
        idx = idx_df[idx_df['h_dong'] == dong].index[0]
        
        xyz = pd.DataFrame()
        delta_h = val_data['REG_DTIME'].unique()[1] - val_data['REG_DTIME'].unique()[0] 
        s = val_data['REG_DTIME'].unique().max() + delta_h
        e = val_data['REG_DTIME'].unique().max() + 24*delta_h

        #print(pd.date_range(s,e ,freq = 'h'))
        xyz.index = pd.date_range(s,e ,freq = 'h')
        xyz['prediction'] = pred['prediction'][idx, : , n]
        prediction_df = pd.concat([prediction_df, xyz])    
```

time_idx 계산을 실수해서 plot이 뒤로 미뤄지는 것인지 확인하기 위해서 prediction을 df로 바꿔서 시간대를 붙여주고 이를 붙여서 원본 데이터와 비교하는 방식으로 변경  
근데 차이는 없음  
